In [ ]:
# Anchor working directory to project root so all relative paths resolve correctly.
# Jupyter kernels default to the notebook's own directory (notebooks/), not the project root.
# This cell must be run first and run only once per session.
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

## Asset Scheduler — Demo Notebook

This notebook mirrors `main.py` exactly, executing each pipeline stage in sequence.
Each section names the function called, the file it lives in, and its inputs and outputs.
Run cells top-to-bottom to reproduce the same output files as `main.py`.

### Call flow
```
① load_config          data_loader.py
② preprocess_data      data_loader.py
③ create_df_impute     data_loader.py
④ distribute_months    model.py          ← CP-SAT solve #1 (month assignment)
⑤ split_solution_to_seasonal_batches    solution_parser.py
⑥ convert_batches_to_mvs               data_loader.py
⑦ solve_assets_for_each_season         solution_parser.py  ← CP-SAT solves #2–5
   └─ [visualise]  read_and_plot_batch_season_solutions    (notebook only)
⑧ create_full_season_solutions         solution_parser.py
⑨ merge_season_solutions               solution_parser.py
⑩ postprocess_data                     solution_parser.py
```

In [ ]:
%matplotlib inline
import calendar
import pickle
import time
import pandas as pd
from IPython.display import display

from scheduler import (
    load_config,
    preprocess_data,
    create_df_impute,
    distribute_months,
    split_solution_to_seasonal_batches,
    convert_batches_to_mvs,
    solve_assets_for_each_season,
    create_full_season_solutions,
    merge_season_solutions,
    postprocess_data,
)
from scheduler.solution_parser import read_and_plot_batch_season_solutions

seasons = ["Summer", "Autumn", "Winter", "Spring"]

---
### ① `load_config()` — `scheduler/data_loader.py`

**Input:** `config.json`  
**Output:** `config` dict — file paths, AOP year, and public holidays parsed into `date` objects

In [ ]:
config = load_config()

print(f"Asset requests filepath: {config['asset_requests_filepath']}")
print(f"Historical file:  {config['historical_durations_filepath']}")
print(f"Relational file:  {config['relational_dependencies_filepath']}")
print(f"AOP year:         {config['aop_year']}")
print(f"Public holidays:  {len(config['public_holidays'])} dates")
print("\nFirst 3 holidays:")
for h in config["public_holidays"][:3]:
    print(f"  {h}")

---
### ② `preprocess_data()` — `scheduler/data_loader.py`

**Input:** `config['asset_requests_filepath']` (Excel)  
**Output:** cleaned DataFrame written **back to the same file** (in-place overwrite)

Removes duplicate asset rows by grouping on `Asset` and concatenating unique values
across all other columns (bundled assets, conflicting assets, etc.).

On first call a raw backup (`_backup.xlsx`) is created alongside the source file.
All subsequent calls preprocess from that backup, so this cell is safe to re-run.

In [ ]:
df_before = pd.read_excel(config["asset_requests_filepath"])
print(f"Rows before preprocessing: {len(df_before)}")
print(f"Unique assets:             {df_before['Asset'].nunique()}")

df_cleaned = preprocess_data(config)

print(f"\nRows after preprocessing:  {len(df_cleaned)}")
print(f"Duplicates removed:        {len(df_before) - len(df_cleaned)}")
print("\nSample output:")
display(df_cleaned.head())

---
### ③ `create_df_impute()` — `scheduler/data_loader.py`

**Inputs:** `historical_durations_filepath`, `relational_dependencies_filepath`, `asset_requests_filepath`  
**Output:** `df_all_impute` — assets joined with historical durations (`HistMedDuration`) and relational metadata

Joins three sources on `Asset` to produce a single enriched DataFrame
used by the month-distribution solver.

In [ ]:
df_all_impute = create_df_impute(
    config["historical_durations_filepath"],
    config["relational_dependencies_filepath"],
    config["asset_requests_filepath"],
)

print(f"Shape: {df_all_impute.shape}")
print(f"Columns: {list(df_all_impute.columns)}")
print()
display(df_all_impute.head())

---
### ④ `distribute_months()` — `scheduler/model.py` — CP-SAT solve #1

**Input:** `df_all_impute`  
**Output:** `solution` — dict keyed by `(asset, month_abbr)` → 0 or 1

Each asset is assigned to exactly one month. Constraints enforce:
- dependency groups land in the same month
- per-month concurrency limits are respected

The objective minimises workload imbalance across months and penalises assignments outside
each asset's historically preferred months.

In [ ]:
t0 = time.time()
solution = distribute_months(df_all_impute)
elapsed = time.time() - t0
print(f"Solve time: {elapsed:.1f}s")

# Month-by-month assignment counts
months = [calendar.month_abbr[i] for i in range(1, 13)]
assets = list(df_all_impute["Asset"])
month_counts = {
    m: sum(1 for a in assets if solution.get((a, m), 0) == 1)
    for m in months
}
month_series = pd.Series(month_counts, name="Assets Assigned")
month_series.index.name = "Month"
print("\nAssets assigned per month:")
display(month_series.to_frame().T)

---
### ⑤ `split_solution_to_seasonal_batches()` — `scheduler/solution_parser.py`

**Inputs:** `df_all_impute`, `solution`, `seasons`, `config`  
**Output:** `df_soln` — DataFrame with `Assigned Month` and `Season` columns  
**Side-effect:** writes `outputs/05_season_batches_{season}.xlsx` for each season

Month → season mapping:  
Jan, Feb, Dec → **Summer** | Mar–May → **Autumn** | Jun–Aug → **Winter** | Sep–Nov → **Spring**

In [ ]:
df_soln = split_solution_to_seasonal_batches(df_all_impute, solution, seasons, config)

print("Season distribution:")
display(df_soln["Season"].value_counts().rename("Assets").to_frame())

print("\nSample rows:")
display(df_soln[["Asset", "Assigned Month", "Season"]].head(10))

print("\nFiles written:")
for s in seasons:
    print(f"  outputs/05_season_batches_{s}.xlsx")

---
### ⑥ `convert_batches_to_mvs()` — `scheduler/data_loader.py`

**Input:** reads `05_season_batches_{season}.xlsx` per season  
**Output:** writes `06_mvs_groups_{season}.pkl` per season — pickled dict of dependency groups

Traverses `Bundled Assets` relationships via BFS to identify root parents and their
full dependency trees. Each MVS (Minimum Viable Set) group stores:
- `group` — all members (parent + dependents)
- `duration` — maximum historical median duration across work members
- `Conflicting Assets` — external assets that must not run concurrently

These groups are the atomic units scheduled in step ⑦.

In [ ]:
convert_batches_to_mvs(seasons)

print("MVS groups per season:")
for s in seasons:
    with open(f"outputs/06_mvs_groups_{s}.pkl", "rb") as f:
        mvs = pickle.load(f)
    print(f"  {s}: {len(mvs)} groups")

# Inspect a sample group from the first season
first_season = seasons[0]
with open(f"outputs/06_mvs_groups_{first_season}.pkl", "rb") as f:
    mvs = pickle.load(f)

sample_key = next(iter(mvs))
print(f"\nSample MVS group ({first_season} — '{sample_key}'):")
print(f"  Group members:      {mvs[sample_key]['group']}")
print(f"  Duration (days):    {mvs[sample_key]['duration']}")
print(f"  Conflicting assets: {mvs[sample_key]['Conflicting Assets']}")

print("\nFiles written:")
for s in seasons:
    print(f"  outputs/06_mvs_groups_{s}.pkl")

---
### ⑦ `solve_assets_for_each_season()` — `scheduler/solution_parser.py` — CP-SAT solves #2–5

**Inputs:** `seasons`, `config`; reads `06_mvs_groups_{season}.pkl` per season  
**Output:** writes `07_day_schedule_{season}.xlsx` per season — day-level schedule (asset × day pairs)

One CP-SAT solve per season. Each MVS group is scheduled across the season's working days,
respecting duration, concurrency limits, and public holidays. The objective balances daily
workload and minimises concurrent starts.

In [ ]:
t0 = time.time()
solve_assets_for_each_season(seasons, config)
elapsed = time.time() - t0
print(f"\nTotal solve time (all seasons): {elapsed:.1f}s")

print("\nSolution sizes:")
for s in seasons:
    sln = pd.read_excel(f"outputs/07_day_schedule_{s}.xlsx")
    n_assets = sln["Asset"].nunique()
    n_pairs = len(sln)
    print(f"  {s}: {n_assets} assets, {n_pairs} asset-day pairs")

print("\nFiles written:")
for s in seasons:
    print(f"  outputs/07_day_schedule_{s}.xlsx")

---
### Visualise seasonal schedules — `read_and_plot_batch_season_solutions()` — `scheduler/solution_parser.py`

Not called in `main.py` — notebook-only visual step (marked ○ in the call flow graph).

Reads the freshly written `07_day_schedule_{season}.xlsx` files and calls `plot_solution()` → `daterange()`
for each season. Each chart shows outage blocks plotted against calendar days, with weekends
(light grey) and public holidays (silver) shaded.

**Saves to:** `notebooks/plots/schedule_{season}_{year}.png`

In [ ]:
read_and_plot_batch_season_solutions(seasons, config, output_dir="notebooks/plots")

print("\nPlots saved:")
for s in seasons:
    print(f"  notebooks/plots/schedule_{s}_{config['aop_year']}.png")

---
### ⑧ `create_full_season_solutions()` — `scheduler/solution_parser.py`

**Inputs:** reads `07_day_schedule_{season}.xlsx` + `06_mvs_groups_{season}.pkl` per season; `config['asset_requests_filepath']`  
**Output:** writes `08b_season_schedule_{season}.xlsx` per season

Merges the day-level schedule with original asset metadata (planned start/finish dates and times).
Expands each MVS group — adding rows for all dependent assets with dates inherited from
their parent. Derives a `Time Frame` column: `Daily` for single-day assets, `Continuous` for multi-day.

In [ ]:
create_full_season_solutions(seasons, config)

print("Output shapes:")
for s in seasons:
    df = pd.read_excel(f"outputs/08b_season_schedule_{s}.xlsx")
    print(f"  08b_season_schedule_{s}.xlsx — {df.shape[0]} rows × {df.shape[1]} cols")

print("\nSample (Summer):")
display(
    pd.read_excel("outputs/08b_season_schedule_Summer.xlsx")
    [["Asset", "Planned Start Date", "Planned Finish Date", "Time Frame"]]
    .head()
)

---
### ⑨ `merge_season_solutions()` — `scheduler/solution_parser.py`

**Input:** reads `08b_season_schedule_{season}.xlsx` for all four seasons  
**Output:** writes `09_draft_aop_{year}_to_{year+1}.xlsx`; returns `df_full_sln`

Concatenates all four seasonal DataFrames, sorts by planned start date, and reformats dates
as `DD/MM/YYYY` strings.

In [ ]:
df_full_sln = merge_season_solutions(seasons, config)

aop_year = config["aop_year"]
print(f"Combined AOP: {df_full_sln.shape[0]} rows × {df_full_sln.shape[1]} cols")
print(f"\nDate format sample (DD/MM/YYYY):")
display(
    df_full_sln[["Asset", "Planned Start Date", "Planned Finish Date", "Time Frame"]]
    .head()
)
print(f"\nWritten to: outputs/09_draft_aop_{aop_year}_to_{aop_year + 1}.xlsx")

---
### ⑩ `postprocess_data()` — `scheduler/solution_parser.py`

**Input:** reads `09_draft_aop_{year}_to_{year+1}.xlsx`  
**Output:** Formatted DataFrame; overwrites same file

Transforms column names to UPPERCASE format and maps abbreviated request reason codes
to full descriptions:

| Code | Maps to |
|------|--------|
| APJ, SEL, CAP | Project |
| PDM, PDM-L, PM, PM Forecast | Station Maintenance |

Adds constant columns `NATURE = "RS"` and `REQUESTOR = ""`.

In [ ]:
transformed_df = postprocess_data(config)

print(f"Output shape: {transformed_df.shape}")
print(f"\nColumns: {list(transformed_df.columns)}")
print("\nSample output:")
display(
    transformed_df[
        ["ASSET", "PLANNED START DATE", "PLANNED FINISH DATE", "REQUEST REASONS", "TIME FRAME"]
    ].head(10)
)